In [6]:
import numpy as np
import pandas as pd
import cvxpy as cp

def run_module_3_allocation(budget_seq, beta_matrix, regions=6, categories=4, T=5):
    """
    MODULE 3: TỐI ƯU PHÂN BỔ (KẾT HỢP BÀI 4 & BÀI 8)
    Phân bổ ngân sách công nghệ số theo Không gian (Vùng) và Thời gian (Năm).
    """
    
    # 1. KHỞI TẠO BIẾN QUYẾT ĐỊNH (FIX CẢNH BÁO DIMENSION)
    # Thay vì dùng biến 3 chiều, ta dùng một list chứa T biến 2 chiều (Hạng mục x Vùng).
    # Việc này giúp CVXPY biên dịch siêu nhanh và không bị văng cảnh báo SCIPY backend.
    X = [cp.Variable((categories, regions), nonneg=True) for _ in range(T)]
    
    constraints = []
    
    # 2. RÀNG BUỘC THEO NĂM & CÔNG BẰNG VÙNG MIỀN
    for t in range(T):
        # Ràng buộc a: Tổng phân bổ trong năm t không vượt quá ngân sách năm đó
        constraints.append(cp.sum(X[t]) <= budget_seq[t])
        
        # Ràng buộc b: Công bằng vùng miền (Mỗi vùng nhận tối thiểu 10% và tối đa 30% tổng NS)
        for r in range(regions):
            constraints.append(cp.sum(X[t][:, r]) >= 0.10 * budget_seq[t])
            constraints.append(cp.sum(X[t][:, r]) <= 0.30 * budget_seq[t])
            
        # Ràng buộc c: Ưu tiên hạng mục (Vd: Hạ tầng số [0] và Nhân lực số [3] tối thiểu 15%)
        constraints.append(cp.sum(X[t][0, :]) >= 0.15 * budget_seq[t])
        constraints.append(cp.sum(X[t][3, :]) >= 0.15 * budget_seq[t])

    # 3. HÀM MỤC TIÊU: TỐI ĐA HÓA TỔNG GDP GAIN CHIẾT KHẤU ĐỘNG
    rho = 0.95  # Hệ số chiết khấu tương lai
    objective_expr = 0
    
    for t in range(T):
        # Tối ưu tốc độ: Dùng hàm cp.multiply để nhân element-wise 2 ma trận thay vì vòng lặp lồng nhau
        objective_expr += (rho**t) * cp.sum(cp.multiply(beta_matrix, X[t]))
                
    objective = cp.Maximize(objective_expr)
    
    # 4. GỌI BỘ GIẢI
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.ECOS, verbose=False)
    
    if prob.status not in [cp.OPTIMAL, "optimal", cp.OPTIMAL_INACCURATE]:
        print("Mô hình thất bại! Hãy kiểm tra lại hệ ràng buộc.")
        return None, 0

    # 5. ĐỊNH DẠNG ĐẦU RA THÀNH DATAFRAME ĐỂ DỄ ĐƯA VÀO DASHBOARD (MODULE 6)
    years = [2026 + t for t in range(T)]
    region_names = [f"Vùng_{r+1}" for r in range(regions)]
    category_names = ["Hạ tầng số (I)", "Chuyển đổi số DN (D)", "Trí tuệ nhân tạo (AI)", "Nhân lực số (H)"]
    
    records = []
    for t in range(T):
        # Rút trích ma trận giá trị của năm t
        val_matrix = X[t].value 
        for j in range(categories):
            for r in range(regions):
                val = val_matrix[j, r]
                if val > 1e-4:  # Lọc bỏ các số tiệm cận 0 do sai số của solver
                    records.append({
                        "Năm": years[t],
                        "Vùng": region_names[r],
                        "Hạng mục": category_names[j],
                        "Ngân sách (Tỷ VND)": val
                    })
                    
    df_results = pd.DataFrame(records)
    return df_results, prob.value

# =====================================================================
# CHẠY THỬ NGHIỆM ĐỘC LẬP (TESTING MODULE 3)
# =====================================================================
if __name__ == "__main__":
    # 1. Tạo dữ liệu giả lập đầu vào (Mock Data)
    T_forecast = 5
    # Ngân sách quốc gia tăng dần qua 5 năm (từ 50.000 đến 70.000 tỷ)
    budget_input = np.array([50000, 55000, 60000, 65000, 70000]) 
    
    # Ma trận Beta: 4 hạng mục x 6 vùng (tạo ngẫu nhiên xoay quanh mức 1.0 - 1.5)
    np.random.seed(42)
    beta_input = np.random.uniform(0.8, 1.5, size=(4, 6))
    
    # 2. Thực thi Module 3
    print("Đang chạy Module 3: Tính toán tối ưu phân bổ Không gian & Thời gian...")
    df_plan, max_gdp_gain = run_module_3_allocation(budget_input, beta_input, T=T_forecast)
    
    # 3. In kết quả chuẩn bị truyền cho Module 6 (Dashboard)
    print("\n" + "="*80)
    print(f"TỔNG GDP GAIN KỲ VỌNG (ĐÃ CHIẾT KHẤU): {max_gdp_gain:,.2f} Tỷ VND")
    print("="*80)
    
    # Tính tổng phân bổ theo năm để kiểm tra độ chính xác của ràng buộc
    summary_by_year = df_plan.groupby("Năm")["Ngân sách (Tỷ VND)"].sum().reset_index()
    print("\nKiểm tra giải ngân theo năm:")
    print(summary_by_year.to_string(index=False))
    
    print("\nBảng phân bổ chi tiết (10 dòng đầu tiên):")
    pd.set_option('display.float_format', lambda x: '%.2f' % x)
    print(df_plan.head(10).to_string(index=False))

Đang chạy Module 3: Tính toán tối ưu phân bổ Không gian & Thời gian...

TỔNG GDP GAIN KỲ VỌNG (ĐÃ CHIẾT KHẤU): 371,057.01 Tỷ VND

Kiểm tra giải ngân theo năm:
 Năm  Ngân sách (Tỷ VND)
2026            50000.00
2027            55000.00
2028            60000.00
2029            65000.00
2030            70000.00

Bảng phân bổ chi tiết (10 dòng đầu tiên):
 Năm   Vùng              Hạng mục  Ngân sách (Tỷ VND)
2026 Vùng_2        Hạ tầng số (I)            15000.00
2026 Vùng_3        Hạ tầng số (I)             2500.00
2026 Vùng_4  Chuyển đổi số DN (D)             5000.00
2026 Vùng_6  Chuyển đổi số DN (D)            15000.00
2026 Vùng_1 Trí tuệ nhân tạo (AI)             5000.00
2026 Vùng_3       Nhân lực số (H)             2500.00
2026 Vùng_5       Nhân lực số (H)             5000.00
2027 Vùng_2        Hạ tầng số (I)            16500.00
2027 Vùng_3        Hạ tầng số (I)             2750.00
2027 Vùng_4  Chuyển đổi số DN (D)             5500.00
